# Week 8: Customer Lifespan Heterogeneity — Why Averages Lie
## Shared Frailty Models

**Masterclass:** [Week 8 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week8_frailty_models.html)

**Dataset:** IBM Telco Customer Churn

**What You'll Build:**
1. Standard Cox PH (Week 2 baseline)
2. Gamma shared frailty model (accounts for unobserved heterogeneity)
3. Compare hazard ratios: frailty-corrected vs. naive
4. Frailty variance interpretation: how much heterogeneity is hidden?

In [ ]:
!pip install lifelines pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df['churned'] = (df['Churn'] == 'Yes').astype(int)

# Features
df['fiber_optic'] = (df['InternetService'] == 'Fiber optic').astype(int)
df['month_to_month'] = (df['Contract'] == 'Month-to-month').astype(int)
df['electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
df['has_tech_support'] = (df['TechSupport'] == 'Yes').astype(int)
df['has_partner'] = (df['Partner'] == 'Yes').astype(int)

covs = ['month_to_month', 'fiber_optic', 'electronic_check',
        'has_tech_support', 'has_partner', 'SeniorCitizen', 'MonthlyCharges']

# Use InternetService as frailty group (shared unobserved heterogeneity within service type)
df['frailty_group'] = df['InternetService'].map(
    {'DSL': 0, 'Fiber optic': 1, 'No': 2})

print(f"Customers: {len(df):,} | Churned: {df['churned'].sum():,}")

---
## Part 2: Standard Cox vs. Frailty Cox

The standard Cox assumes all heterogeneity is captured by covariates. The frailty model adds a random effect for unobserved differences between groups.

In [ ]:
# Standard Cox
cph_standard = CoxPHFitter(penalizer=0.01)
cph_standard.fit(df[covs + ['tenure', 'churned']],
                 duration_col='tenure', event_col='churned')

# Cox with shared frailty (lifelines doesn't have native frailty,
# so we demonstrate via stratification + random effects approximation)
# We compare stratified vs unstratified to show heterogeneity impact

cph_stratified = CoxPHFitter(penalizer=0.01)
cph_stratified.fit(df[covs + ['tenure', 'churned', 'InternetService']],
                   duration_col='tenure', event_col='churned',
                   strata=['InternetService'])

# Compare HRs
print("=== Hazard Ratio Comparison ===")
print(f"{'Covariate':<20} {'Standard':>10} {'Stratified':>10} {'Change':>10}")
print("-" * 52)
for var in covs:
    hr_std = np.exp(cph_standard.params_[var])
    hr_strat = np.exp(cph_stratified.params_[var])
    change = (hr_strat - hr_std) / hr_std * 100
    print(f"{var:<20} {hr_std:>10.3f} {hr_strat:>10.3f} {change:>+9.1f}%")

In [ ]:
# Visualize the difference
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

cph_standard.plot(ax=axes[0])
axes[0].set_title('Standard Cox PH', fontsize=13, fontweight='bold')
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)

cph_stratified.plot(ax=axes[1])
axes[1].set_title('Stratified Cox (Frailty Proxy)', fontsize=13, fontweight='bold')
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

---
## Part 3: Simulating True Gamma Frailty

To demonstrate the frailty concept directly, we simulate data where customers have unobserved "fragility" drawn from a Gamma distribution.

In [ ]:
# Simulation: Gamma frailty
np.random.seed(42)
n = 5000
theta = 0.5  # frailty variance

# Gamma-distributed frailty: mean=1, variance=theta
frailty = np.random.gamma(shape=1/theta, scale=theta, size=n)

# Generate survival times with frailty
baseline_hazard = 0.02
X1 = np.random.binomial(1, 0.5, n)  # binary covariate
true_beta = 0.8

hazard = frailty * baseline_hazard * np.exp(true_beta * X1)
T = np.random.exponential(1/hazard)
C = np.random.uniform(0, 100, n)  # censoring
observed_T = np.minimum(T, C)
event = (T <= C).astype(int)

sim_df = pd.DataFrame({'duration': observed_T, 'event': event, 'X1': X1})

# Fit WITHOUT frailty
cph_no_frailty = CoxPHFitter()
cph_no_frailty.fit(sim_df, 'duration', 'event')

print(f"True beta: {true_beta:.3f}")
print(f"Estimated (no frailty): {cph_no_frailty.params_['X1']:.3f}")
print(f"Attenuation: {(1 - cph_no_frailty.params_['X1']/true_beta)*100:.1f}%")
print(f"\nKey insight: ignoring frailty attenuates coefficients toward zero.")
print(f"The true effect is STRONGER than what standard Cox estimates.")

---
## Exercises

### Exercise 1: Frailty by Contract Type
Use `Contract` as the frailty grouping variable. How do HRs change compared to `InternetService`?

### Exercise 2: Vary Frailty Variance
In the simulation (Part 3), try theta = 0.1, 0.5, 1.0, 2.0. Plot attenuation vs. frailty variance.

### Exercise 3: Business Implication
If the true HR for `electronic_check` is 1.8 but standard Cox shows 1.4, what's the revenue impact of under-estimating? Compute the difference in predicted churn at 12 months.

---
## Key Takeaways
1. **Unobserved heterogeneity** attenuates hazard ratios toward 1.0
2. **Frailty models** recover truer effect sizes
3. **Frailty variance** tells you how much hidden heterogeneity exists
4. **Practical impact**: underestimated HRs lead to under-investment in retention